# 📘 Notebook Thực hành — Chủ đề 4: Xử lý ngoại lệ

**Học phần:** Lập trình Python cho học máy (Khoa Công nghệ Thông tin — Trường Đại học Nha Trang)  
**Tài liệu tham khảo chính:** CS50's Introduction to Programming with Python — *Lecture 3: Exceptions* (Harvard OpenCourseWare)  
🔗 https://cs50.harvard.edu/python/notes/3/

> ℹ️ **Ghi chú kết nối với Chủ đề 3:** Ở Chủ đề 3, để xác thực đầu vào người dùng (ví dụ, yêu cầu nhập một số dương), ta dùng khuôn mẫu `while True: ... if ...: break`. Cách này xử lý tốt trường hợp người dùng nhập **một số không thoả điều kiện** (ví dụ số âm). Nhưng nếu người dùng gõ `"hai"` thay vì `2` thì sao? Dòng `int(input(...))` sẽ ngay lập tức làm chương trình **sụp đổ (crash)** với một thông báo lỗi khó hiểu — vì `while`/`if` không thể "bắt" được kiểu lỗi này! Chủ đề 4 sẽ trang bị cho bạn công cụ mạnh mẽ và "đúng chuẩn Python" hơn để xử lý triệt để những lỗi phát sinh ngay khi chương trình đang chạy — gọi là **ngoại lệ (exceptions)** — thông qua cấu trúc `try`/`except`.

## 🎯 Mục tiêu học tập (CLO a, e)
Sau khi hoàn thành notebook này, sinh viên có thể:
- **4.1** Giải thích khái niệm **ngoại lệ (exception)**; phân biệt **lỗi cú pháp (syntax error)** — phát sinh *trước khi* chương trình chạy — với **lỗi thời gian chạy (runtime error)** — phát sinh *trong lúc* chương trình đang thực thi
- **4.2** Nhận diện một số **loại ngoại lệ** phổ biến trong Python (`ValueError`, `NameError`, `TypeError`, `ZeroDivisionError`, `IndexError`...) và giải thích nguyên nhân phát sinh của từng loại
- **4.3** Sử dụng `try`, `except`, `else`, và `pass` để xử lý ngoại lệ một cách an toàn; kết hợp với vòng lặp và hàm để xây dựng các hàm nhập liệu "phòng thủ" (defensive), không sụp đổ khi người dùng nhập sai

## 📝 Hướng dẫn sử dụng
1. Đọc lý thuyết (ô Markdown), chạy các ô code mẫu bằng **Shift + Enter**.
2. Làm các mục **✏️ Bài tập** ngay trong ô `# TODO` phía dưới mỗi phần.
3. ⚠️ Một số ô code trong notebook này **cố ý** gây lỗi để minh hoạ. Khác với vòng lặp vô hạn ở Chủ đề 3 (phải tuyệt đối tránh chạy), các lỗi ở đây (`SyntaxError`, `ValueError`, `NameError`...) đều khiến ô code **dừng ngay lập tức** với một thông báo lỗi (traceback) — hoàn toàn an toàn để chạy thử và quan sát, không làm treo notebook.
4. Notebook giả định bạn đã hoàn thành Chủ đề 1, 2 và 3 (biết `input`, `print`, hàm `def`, `if/elif/else`, `while`, `for`, `list`).

## 4.1. Khái niệm về ngoại lệ (Exceptions)

Về bản chất, **ngoại lệ (exceptions)** là những vấn đề nảy sinh **trong lúc chương trình của bạn đang chạy (running)** — nói cách khác, đó là những điều "đi sai" bên trong code *trong khi nó đang thực thi*.

Trước khi tìm hiểu sâu về ngoại lệ, hãy phân biệt nó với một loại lỗi khác mà có thể bạn đã từng gặp: **lỗi cú pháp (syntax error)**.

Trong terminal, gõ `code hello.py` để tạo file mới, rồi nhập đoạn code sau (**cố ý** chứa lỗi):

In [1]:
print("hello, world)

SyntaxError: unterminated string literal (detected at line 1) (1807429930.py, line 1)

Lưu ý rằng ta **cố ý** bỏ sót một dấu ngoặc kép đóng ở cuối chuỗi.

Chạy ô code trên (hoặc `python hello.py` trong terminal), trình biên dịch (interpreter) sẽ báo lỗi **`SyntaxError`**. Lỗi cú pháp thường có nghĩa: bạn nên kiểm tra lại xem mình đã gõ code **đúng** hay chưa (thiếu dấu ngoặc, dấu hai chấm, thụt lề sai, v.v.).

> 💡 **Lưu ý quan trọng:** `SyntaxError` xảy ra khi Python cố gắng **đọc (parse)** code của bạn **trước khi** chạy bất kỳ dòng nào — vì vậy, nói một cách chặt chẽ, đây **chưa hẳn** là một "ngoại lệ" đúng nghĩa (ngoại lệ xảy ra "trong lúc chương trình đang chạy"). Tuy nhiên, đây vẫn là loại lỗi phổ biến nhất mà người mới học gặp phải, nên ta xem xét trước để làm phép so sánh với lỗi thời gian chạy dưới đây.

📚 Tìm hiểu thêm: [Python's documentation of Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html)

### Lỗi thời gian chạy (Runtime Errors)

**Lỗi thời gian chạy (runtime error)** là lỗi phát sinh do hành vi không mong đợi **trong lúc** code đang thực thi. Ví dụ: bạn muốn người dùng nhập một số, nhưng họ lại nhập một chữ hoặc ký tự. Chương trình của bạn có thể "ném" (throw) lỗi vì đầu vào không như mong đợi này.

Trong terminal, gõ `code number.py`. Trong text editor, viết code như sau:

In [3]:
x = int(input("What's x? "))
print(f"x is {x}")

KeyboardInterrupt: Interrupted by user

Lưu ý chữ `f` đứng trước chuỗi: nó báo cho Python "nội suy" (interpolate) — chèn trực tiếp giá trị nằm trong `{}` (ở đây là giá trị của biến `x`) vào chuỗi.

Hãy chạy thử ô trên: nhập **`50`** — chương trình chạy đúng như mong đợi. Bây giờ, chạy lại và nhập **`cat`** (thay vì một số) — bạn sẽ thấy:

```
ValueError: invalid literal for int() with base 10: 'cat'
```

Nói cách khác: hàm `int()` **không thể** chuyển đổi văn bản `"cat"` thành một con số nguyên.

Là người lập trình, ta nên có tư duy **"phòng thủ" (defensive programming)**: luôn giả định người dùng **có thể** nhập sai — kể cả gõ chữ, ký tự đặc biệt, hoặc thậm chí không gõ gì cả (chỉ nhấn Enter) — và chuẩn bị sẵn cách xử lý cho những tình huống đó, thay vì để chương trình sụp đổ giữa đường.

Một chiến lược hiệu quả để khắc phục lỗi tiềm ẩn này là tạo ra cơ chế **"xử lý lỗi" (error handling)** để đảm bảo người dùng hành xử đúng như ta mong đợi — đây chính là nội dung mục 4.3 của chủ đề này.

📚 Tìm hiểu thêm: [Python's documentation of Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html)

**✏️ Bài tập 1 — Phân biệt lỗi cú pháp và lỗi thời gian chạy:**
1. Trong ô code dưới, cố ý viết một đoạn code có **lỗi cú pháp** (ví dụ: thiếu dấu `:` sau `if`, hoặc thiếu dấu đóng ngoặc `)`). Chạy ô để quan sát thông báo `SyntaxError`, sau đó **sửa lại cho đúng** và chạy lại để ô thực thi thành công.
2. Trong ô code tiếp theo, viết một đoạn code **không có lỗi cú pháp**, nhưng **có thể** gây **lỗi thời gian chạy** khi chạy với input không phù hợp: xin người dùng nhập hai số nguyên `a` và `b` (dùng `int(input(...))`), rồi in ra kết quả `a / b`. Chạy thử với `b` khác `0` (chạy đúng), sau đó chạy lại và nhập `b = 0` để quan sát traceback lỗi `ZeroDivisionError`.
3. Thêm một dòng **comment** (`#`) ở đầu mỗi ô, giải thích ngắn gọn: lỗi ở bước 1 và lỗi ở bước 2 khác nhau ở điểm nào về **thời điểm** chúng xuất hiện (trước khi chạy hay trong lúc chạy)?

In [ ]:
# TODO 1: Viết một đoạn code có lỗi cú pháp, quan sát SyntaxError, rồi sửa lại cho đúng
# (Giải thích ngắn gọn ở đây về thời điểm lỗi xuất hiện)



In [ ]:
# TODO 2: Viết chương trình chia hai số nguyên do người dùng nhập, gây ZeroDivisionError khi b = 0
# (Giải thích ngắn gọn ở đây về thời điểm lỗi xuất hiện)



## 4.2. Các loại ngoại lệ (Types of Exceptions)

Mỗi loại lỗi trong Python được đại diện bởi một **lớp ngoại lệ (exception class)** riêng, giúp ta biết chính xác điều gì đã xảy ra. Dưới đây là một số **loại ngoại lệ phổ biến (built-in exceptions)** mà bạn sẽ thường gặp:

| Loại ngoại lệ | Khi nào xảy ra | Ví dụ |
|---|---|---|
| `SyntaxError` | Code viết sai cú pháp — Python không thể "đọc hiểu" trước khi chạy | `print("hi)` |
| `ValueError` | Giá trị có kiểu dữ liệu đúng nhưng nội dung không hợp lệ cho phép toán | `int("cat")` |
| `TypeError` | Thực hiện phép toán/hàm trên (các) kiểu dữ liệu không phù hợp | `"3" + 3` |
| `ZeroDivisionError` | Chia một số cho `0` | `10 / 0` |
| `NameError` | Tham chiếu đến một biến **chưa từng được gán giá trị** | `print(x)` khi `x` chưa được gán |
| `IndexError` | Truy cập chỉ số (index) **không tồn tại** trong `list` | `[1, 2, 3][10]` |
| `KeyError` | Truy cập khoá (key) **không tồn tại** trong `dict` | `{"a": 1}["b"]` |
| `FileNotFoundError` | Mở một file **không tồn tại** | `open("khongtontai.txt")` |

> 📚 Nguồn tổng hợp: [Python's Built-in Exceptions documentation](https://docs.python.org/3/library/exceptions.html)

**✏️ Bài tập 2 — Nhận diện loại ngoại lệ:** Với mỗi ô code dưới đây, **trước khi chạy**, hãy viết một dòng comment (`#`) dự đoán loại ngoại lệ (exception) sẽ xảy ra (tham khảo bảng trên). Sau đó chạy ô để kiểm tra dự đoán của bạn có đúng không.
- a) `print(ten)` (biến `ten` chưa từng được gán ở bất kỳ đâu trước đó)
- b) `"5" + 5`
- c) `[10, 20, 30][10]`
- d) `10 / 0`

In [ ]:
# TODO a: Dự đoán loại ngoại lệ (viết comment), rồi chạy để kiểm tra
print(ten)

In [ ]:
# TODO b: Dự đoán loại ngoại lệ (viết comment), rồi chạy để kiểm tra
"5" + 5

In [ ]:
# TODO c: Dự đoán loại ngoại lệ (viết comment), rồi chạy để kiểm tra
[10, 20, 30][10]

In [ ]:
# TODO d: Dự đoán loại ngoại lệ (viết comment), rồi chạy để kiểm tra
10 / 0

Trong Python, `try` và `except` là cách để **"thử" (try)** một đoạn code có khả năng gây lỗi trước khi nó thực sự trở thành sự cố không kiểm soát được. Hãy quay lại ví dụ `number.py` và thử áp dụng:

In [4]:
try:
    x = int(input("What's x? "))
    print(f"x is {x}")
except ValueError:
    print("x is not an integer")

x is 10


Chạy thử ô trên: nhập `50` sẽ được chấp nhận và in ra kết quả bình thường. Nhập `cat` sẽ hiển thị thông báo lỗi **thân thiện** (`"x is not an integer"`) cho người dùng, thay vì một traceback khó hiểu.

Tuy nhiên, đây **vẫn chưa** là cách viết tốt nhất. Nhận thấy ta đang `try` **hai dòng** code. Theo thông lệ tốt (best practice), ta nên chỉ `try` **số dòng ít nhất có thể** — tức là chỉ những dòng mà ta thật sự lo ngại có khả năng gây lỗi. Ở đây, chỉ dòng `int(input(...))` mới có khả năng gây lỗi; dòng `print()` thì không. Hãy điều chỉnh code:

In [6]:
try:
    x = int(input("What's x? "))
except ValueError:
    print("x is not an integer")
print(f"x is {x}")

x is not an integer
x is 32


Đoạn code trên đạt được mục tiêu "`try` ít dòng nhất có thể" — **nhưng**, nếu bạn chạy và nhập `cat`, bạn sẽ gặp phải một **lỗi hoàn toàn mới**:

```
NameError: name 'x' is not defined
```

Hãy dừng lại và suy nghĩ: **tại sao** `x` lại "không được định nghĩa" trong một số trường hợp?

Hãy xem xét **thứ tự thực hiện các phép toán (order of operations)** trong dòng `x = int(input("What's x?"))`: Python thực hiện từ **phải sang trái** — trước tiên gọi `input()` để lấy chuỗi người dùng nhập vào, sau đó gọi `int(...)` để chuyển chuỗi đó thành số nguyên, và **chỉ sau khi** `int()` thành công, giá trị mới được **gán (`=`)** vào biến `x` ở bên trái.

Nếu `int("cat")` gây lỗi (ném ra `ValueError`) ngay tại bước chuyển đổi, thì phép gán (`=`) sẽ **không bao giờ xảy ra** — nghĩa là biến `x` **chưa từng được tạo ra**! Do đó, khi chương trình chạy tiếp đến dòng `print(f"x is {x}")` ở cuối, Python không tìm thấy biến `x` nào cả → `NameError`.

👉 **Bài học quan trọng:** hai cách viết code tưởng chừng "cải tiến" nhau lại có thể dẫn đến **hai loại ngoại lệ khác nhau** (`ValueError` so với `NameError`) — hiểu đúng **nguyên nhân** của từng loại lỗi là chìa khoá để viết code xử lý ngoại lệ đúng đắn. Mục 4.3 tiếp theo sẽ giới thiệu cách giải quyết triệt để vấn đề này bằng `else`.

**✏️ Bài tập 3 — Giải thích lỗi `NameError` (câu hỏi khái niệm):** Dựa vào đoạn code ở ô ngay trên (phiên bản `try` chỉ có 1 dòng), hãy giải thích **bằng lời của riêng bạn**, viết vào ô Markdown ngay dưới đây, tại sao `x` "không được định nghĩa" khi người dùng nhập `cat`.

💡 *Gợi ý:* hãy liệt kê **từng bước** Python thực hiện trong dòng `x = int(input("What's x?"))` khi input là `"50"` (thành công) so với khi input là `"cat"` (lỗi) — bước nào xảy ra trước, bước nào xảy ra sau, và bước gán (`=`) có luôn được thực hiện không?

✍️ **Trả lời của bạn:** *(nhấp đúp chuột vào ô này để chỉnh sửa và viết câu trả lời)*

_...(viết giải thích của bạn ở đây)..._

## 4.3. Xử lý ngoại lệ với try-except

### `else`

Hoá ra Python có một cách khác để triển khai `try` có thể "bắt" được lỗi `NameError` vừa gặp ở trên một cách gọn gàng: sử dụng mệnh đề **`else`** đi kèm `try`/`except`.

`else` hoạt động như sau: **nếu không có ngoại lệ nào xảy ra** trong khối `try` (nghĩa là dòng `x = int(input(...))` chạy **thành công**), Python sẽ chạy khối lệnh bên trong `else`. Vì `else` chỉ chạy khi `try` **thành công**, biến `x` **chắc chắn** đã được gán giá trị — không còn nguy cơ `NameError` nữa!

Chạy `python number.py` (hoặc ô trên) và nhập `50`: kết quả sẽ được in ra (trong khối `else`). Chạy lại và nhập `cat`: chương trình sẽ **bắt được lỗi** (trong khối `except`), và khối `else` sẽ **không** chạy — do đó không có `NameError`.

Hãy cải thiện code thêm một bước: nhận thấy ta đang hơi "thô lỗ" với người dùng — nếu họ không hợp tác (nhập sai), ta hiện tại chỉ đơn giản... **kết thúc chương trình**. Hãy dùng vòng lặp (đã học ở Chủ đề 3!) để hỏi lại `x` liên tục cho đến khi họ hợp tác:

In [9]:
while True:
    try:
        x = int(input("What's x? "))
    except ValueError:
        print("x is not an integer")
    else:
        break

print(f"x is {x}")

x is not an integer
x is not an integer
x is not an integer
x is not an integer
x is not an integer
x is not an integer
x is 32


`while True` sẽ lặp **vô hạn** — nhưng lần này **an toàn**, vì có `break` bên trong `else`: hễ người dùng nhập đúng, chương trình `break` thoát vòng lặp **ngay**, rồi in kết quả. Nếu nhập sai, không có `break` nào được thực thi, vòng lặp tự động lặp lại và hỏi người dùng nhập `x` một lần nữa.

**✏️ Bài tập 4 — Xin số dương an toàn:** Viết chương trình (chưa cần đóng gói thành hàm ở bài này) theo các yêu cầu:
1. Dùng `while True` kết hợp `try`/`except ValueError`/`else` để hỏi người dùng nhập một số nguyên (giống ví dụ trên).
2. Bên trong `else`, dùng `if` để kiểm tra xem số đó **có lớn hơn `0`** không:
   - Nếu **có**: in `"Bạn đã nhập: X"` (với `X` là số đã nhập) và `break`.
   - Nếu **không** (số `<= 0`): in `"Số phải lớn hơn 0, vui lòng nhập lại!"` (không cần `break` — vòng lặp sẽ tự lặp lại).

In [ ]:
# TODO: Viết chương trình xin số dương an toàn (while + try/except/else + if)



### Tạo hàm để lấy một số nguyên (`get_int`)

Chắc hẳn sẽ có rất nhiều lúc ta cần lấy một số nguyên hợp lệ từ người dùng. Hãy **trừu tượng hoá (abstract)** khả năng này thành một hàm riêng, có thể tái sử dụng ở bất cứ đâu trong chương trình:

In [ ]:
def main():
    x = get_int()
    print(f"x is {x}")


def get_int():
    while True:
        try:
            x = int(input("What's x? "))
        except ValueError:
            print("x is not an integer")
        else:
            break
    return x


main()

Nhận thấy ta đã đạt được nhiều tính chất tốt: đầu tiên, ta đã **trừu tượng hoá** khả năng "lấy một số nguyên hợp lệ" ra khỏi `main`. Giờ đây, toàn bộ chương trình (nhìn từ `main`) chỉ còn đúng **3 dòng** — mọi phần phức tạp về xác thực đầu vào đã được "giấu" gọn gàng bên trong `get_int`.

Ta có thể cải thiện chương trình này thêm nữa. Hãy suy nghĩ: còn cách nào khác để cải thiện không?

In [ ]:
def main():
    x = get_int()
    print(f"x is {x}")


def get_int():
    while True:
        try:
            x = int(input("What's x? "))
        except ValueError:
            print("x is not an integer")
        else:
            return x


main()

Nhận thấy `return` không chỉ **trả về một giá trị** — nó còn có khả năng **thoát ngay khỏi vòng lặp** (giống `break`), **đồng thời** thoát khỏi chính hàm đang chứa nó, tất cả trong một hành động duy nhất.

Một số người có thể lập luận ta có thể viết gọn hơn nữa như sau:

In [ ]:
def main():
    x = get_int()
    print(f"x is {x}")


def get_int():
    while True:
        try:
            return int(input("What's x? "))
        except ValueError:
            print("x is not an integer")


main()

Nhận thấy đoạn code này thực hiện **chính xác điều tương tự** như phiên bản trước, chỉ với **ít dòng hơn**: nếu `int(input(...))` thành công, `return` ngay lập tức giá trị đó (không cần biến trung gian `x`, và không cần `else` — vì `return` đã tự động "thoát" khỏi hàm ngay khi không có lỗi xảy ra).

### `pass`

Ta có thể làm cho code **không cảnh báo** người dùng bằng thông báo lỗi, mà chỉ đơn giản **hỏi lại** câu hỏi ban đầu một cách im lặng. Hãy sửa code như sau:

In [ ]:
def main():
    x = get_int()
    print(f"x is {x}")


def get_int():
    while True:
        try:
            return int(input("What's x? "))
        except ValueError:
            pass


main()

Nhận thấy code vẫn hoạt động đúng, nhưng sẽ **không** còn liên tục hiển thị thông báo lỗi cho người dùng nữa. `pass` là một từ khoá đặc biệt trong Python, có nghĩa: **"không làm gì cả"**. Nó cho phép ta viết một khối lệnh (như `except`) mà **không cần có nội dung thực thi nào** bên trong — vì về mặt cú pháp, Python **yêu cầu** phải có ít nhất một dòng lệnh sau dấu `:`, `except:` không thể để hoàn toàn trống.

Trong một số trường hợp, bạn sẽ muốn thông báo **thật rõ** cho người dùng biết lỗi gì đã xảy ra (dùng `print(...)` trong `except`). Trong trường hợp khác, bạn có thể chỉ đơn giản muốn hỏi lại mà không cần giải thích (dùng `pass`). Tuỳ vào ngữ cảnh ứng dụng thực tế mà bạn chọn cách phù hợp.

📚 Tìm hiểu thêm: [Python's documentation of `pass`](https://docs.python.org/3/reference/simple_stmts.html#the-pass-statement)

Một cải tiến cuối cùng: hiện tại, lời nhắc (`"What's x? "`) đang bị **"hard-code"** (viết cứng) ngay trong `get_int`. Trong thực tế, ta thường muốn **truyền vào** một lời nhắc tuỳ ý mà người dùng sẽ nhìn thấy — tuỳ vào chương trình đang gọi hàm này. Hãy sửa code lần cuối:

In [ ]:
def main():
    x = get_int("What's x? ")
    print(f"x is {x}")


def get_int(prompt):
    while True:
        try:
            return int(input(prompt))
        except ValueError:
            pass


main()

Giờ đây, `get_int(prompt)` là một hàm **tổng quát (general-purpose)**, có thể tái sử dụng với **bất kỳ lời nhắc nào** ta muốn, ở bất cứ chương trình nào cần lấy một số nguyên hợp lệ từ người dùng. Đây chính là một **mẫu hình (pattern)** rất phổ biến mà bạn sẽ còn dùng lại nhiều lần trong suốt học phần này.

**✏️ Bài tập 5 — Viết hàm `get_float(prompt)`:**
1. Viết hàm `get_float(prompt)` hoàn toàn tương tự `get_int(prompt)` (dùng `while True`, `try`, `except ValueError`, `pass`, tham số hoá `prompt`) nhưng dùng `float(...)` thay vì `int(...)` để cho phép người dùng nhập số thực (ví dụ `3.14`).
2. Viết hàm `main()`: dùng `get_float(...)` **hai lần** để hỏi `"Chiều dài: "` và `"Chiều rộng: "` của một hình chữ nhật, rồi tính và in diện tích (`dài × rộng`).
3. Gọi `main()` ở cuối file.

In [ ]:
# TODO: Viết get_float(prompt) và main() để tính diện tích hình chữ nhật



**🏁 Bài tập tổng hợp cuối Chủ đề 4 — "Sổ điểm an toàn" (Safe Gradebook)**

Đây là bài toán kết hợp gần như **toàn bộ** nội dung Chủ đề 4 (`try`/`except`/`pass`) với kiến thức Chủ đề 3 (`while`, `list`, `len()`).

**Yêu cầu:**
1. Viết hàm `get_int(prompt)` chuẩn (dùng `while True`, `try`, `except ValueError`, `pass`, tham số hoá `prompt`) — có thể tái sử dụng nguyên bản từ mục 4.3.
2. Viết hàm `main()`:
   - Tạo một `list` rỗng: `scores = []`.
   - Dùng `while True` để liên tục:
     - Dùng `input()` **trực tiếp** (không dùng `get_int`, vì ở đây ta cần xử lý riêng chuỗi đặc biệt `"done"`) với lời nhắc `"Nhập điểm (hoặc gõ 'done' để kết thúc): "`.
     - Nếu chuỗi nhập vào **đúng bằng** `"done"` (chữ thường), `break`.
     - Ngược lại, dùng `try`/`except ValueError` để cố gắng chuyển chuỗi đó thành `float`:
       - Nếu **thành công**: thêm giá trị đó vào `scores`.
       - Nếu **lỗi** (`ValueError`): dùng `pass` (bỏ qua, vòng lặp tự động hỏi lại).
   - Sau khi thoát vòng lặp: nếu `len(scores) > 0`, tính **tổng** (cộng dồn bằng vòng lặp `for`, như đã học ở Chủ đề 3) và in ra **điểm trung bình**; nếu `scores` rỗng, in `"Không có điểm nào được nhập."`.
3. Gọi `main()` ở cuối file.

**Ví dụ chạy thử:**
```
Nhập điểm (hoặc gõ 'done' để kết thúc): 8
Nhập điểm (hoặc gõ 'done' để kết thúc): abc
Nhập điểm (hoặc gõ 'done' để kết thúc): 9.5
Nhập điểm (hoặc gõ 'done' để kết thúc): done
Điểm trung bình: 8.75
```

💡 *Gợi ý thêm phần tử vào `list`:*
- **Cách 1 (khuyến khích):** `scores.append(gia_tri)` — phương thức `append()` có sẵn của `list`, thêm một phần tử vào **cuối** danh sách (bạn sẽ tìm hiểu kỹ các phương thức của `list` ở chủ đề sau, nhưng có thể dùng thử ngay bây giờ!).
- **Cách 2 (chỉ dùng kiến thức đã học):** `scores = scores + [gia_tri]` — nối (bằng phép `+`) một `list` chỉ chứa 1 phần tử mới vào cuối `scores`.

In [ ]:
# TODO: Viết chương trình "Sổ điểm an toàn" (get_int, main với while/try/except/pass/list)



## ✅ Tổng kết Chủ đề 4

Qua notebook này, bạn đã được học và thực hành:

**4.1 — Khái niệm về ngoại lệ:**
- Ngoại lệ (exceptions) là các vấn đề phát sinh **trong lúc** chương trình đang chạy
- Phân biệt lỗi cú pháp (`SyntaxError`, phát sinh **trước** khi chạy) với lỗi thời gian chạy (runtime error, phát sinh **trong lúc** chạy)
- Tư duy lập trình "phòng thủ" (defensive programming)

**4.2 — Các loại ngoại lệ:**
- Bảng tổng hợp các ngoại lệ phổ biến: `ValueError`, `TypeError`, `ZeroDivisionError`, `NameError`, `IndexError`, `KeyError`, `FileNotFoundError`
- Hiểu *nguyên nhân* gốc rễ của lỗi (ví dụ: thứ tự thực hiện phép toán từ phải sang trái dẫn đến `NameError` khi `try` chỉ có 1 dòng)

**4.3 — Xử lý ngoại lệ với `try`-`except`:**
- Cấu trúc `try`/`except` cơ bản, nguyên tắc "`try` ít dòng nhất có thể"
- Mệnh đề `else` — chỉ chạy khi `try` **không** gặp lỗi, giải quyết triệt để vấn đề biến chưa được gán
- Kết hợp `while True` + `try`/`except`/`else` + `break` để **hỏi lại** người dùng cho đến khi nhập đúng
- Trừu tượng hoá thành hàm `get_int()`, dùng `return` thay cho `break`
- Từ khoá `pass` — "không làm gì cả", dùng khi muốn xử lý ngoại lệ một cách im lặng
- Tham số hoá lời nhắc (`prompt`) để hàm `get_int(prompt)` trở nên tổng quát, tái sử dụng được

### 📚 Tài liệu tham khảo thêm
- CS50P Lecture 3 — Exceptions: https://cs50.harvard.edu/python/notes/3/
- Python docs — Errors and Exceptions: https://docs.python.org/3/tutorial/errors.html
- Python docs — Built-in Exceptions: https://docs.python.org/3/library/exceptions.html
- Python docs — `pass` statement: https://docs.python.org/3/reference/simple_stmts.html#the-pass-statement

### ➡️ Bước tiếp theo
Chủ đề 5 sẽ giới thiệu về **Thư viện và module (Libraries)** trong Python — cách cài đặt, `import` và sử dụng các thư viện có sẵn (chuẩn của Python như `os`, `sys`, `math`, `random`, `re`, cũng như thư viện của bên thứ ba từ PyPI) để không phải "tự phát minh lại bánh xe" cho những bài toán đã có giải pháp sẵn. Hãy hoàn thành đầy đủ các bài tập ✏️ trong notebook này trước khi chuyển sang chủ đề tiếp theo!